In [1]:
import sys

# MCP launches its servers as subprocesses and hands them this notebook's
# stderr, which must have a real file descriptor. Notebook kernels do not
# provide one, so we point stderr at a log file. The MCP library captures
# stderr the moment it first loads, so this redirect must run before any
# MCP import, which is why it sits at the top of the setup cell.
sys.stderr = open("mcp_server.log", "w")

In [2]:
!pip install -q langchain langchain-mcp-adapters mcp fastmcp

In [3]:
%%writefile math_server.py
from fastmcp import FastMCP

mcp = FastMCP("Math")

@mcp.tool()
def safe_calculator(expression: str) -> str:
    """Evaluate a basic math expression, such as '8273 * 1821'.
    Supports +, -, *, /, parentheses, and numbers only."""
    try:
        result = eval(expression, {"__builtins__": None}, {})
        return str(result)
    except Exception as e:
        return f"Error: {e}"

if __name__ == "__main__":
    mcp.run(transport="stdio")

Overwriting math_server.py


In [4]:
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient({
    "math": {
        "transport": "stdio",
        "command": "python",
        "args": ["math_server.py"],
    },
})

tools = await client.get_tools()
for t in tools:
    print(f"{t.name}: {t.description}")

safe_calculator: Evaluate a basic math expression, such as '8273 * 1821'.
Supports +, -, *, /, parentheses, and numbers only.


In [5]:
%%writefile trip_server.py
# Server preview only: not executed in this activity.
from fastmcp import FastMCP

mcp = FastMCP("TripUtilities")

@mcp.tool()
def estimate_daily_budget(days: int, daily_cost: float) -> float:
    """Estimate a trip budget from duration and daily cost."""
    if days < 0 or daily_cost < 0:
        raise ValueError("Inputs must be nonnegative")
    return days * daily_cost

if __name__ == "__main__":
    mcp.run(transport="stdio")

Overwriting trip_server.py


In [6]:
import sys
from pathlib import Path
from fastmcp import Client
from fastmcp.client.transports import StdioTransport

transport = StdioTransport(
    command=sys.executable,
    args=[str(Path("trip_server.py").resolve())]
)

client = Client(transport)

async with client:
    # Discover tools
    tools = await client.list_tools()
    print("Available tools:", [t.name for t in tools])

    # Call tool through MCP
    result = await client.call_tool(
        "estimate_daily_budget",
        {"days": 5, "daily_cost": 120.0}
    )

    print("Budget:", result.data)

Available tools: ['estimate_daily_budget']
Budget: 600.0
